# Data Cleaning

Data cleaning involves identifying and removing any missing, duplicate or irrelevant data.

Raw data (log file, transactions, audio /video recordings, etc) is often noisy, incomplete and inconsistent which can negatively impact the accuracy of the model.
The goal of data cleaning is to ensure that the data is accurate, consistent and free of errors.
Clean datasets are also important in EDA (Exploratory Data Analysis), which enhances the interpretability of data so that the right actions can be taken based on insights.

# How to Perform Data Cleaning

Data cleaning involves identifying issues like missing values, duplicates, and outliers, followed by applying appropriate techniques to fix them. The following steps are essential to perform data cleaning:

- Remove Unwanted Observations: Eliminate duplicates, irrelevant entries or redundant data that add noise.

- Fix Structural Errors: Standardize data formats and variable types for consistency.

- Manage Outliers: Detect and handle extreme values that can skew results, either by removal or transformation.

- Handle Missing Data: Address gaps using imputation, deletion or advanced techniques to maintain accuracy and integrity.

## Implementation for Data Cleaning

Let's understand each step for Database Cleaning using titanic dataset.

### Step 1: Import Libraries and Load Dataset

We will import all the necessary libraries i.e pandas and numpy.

In [92]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Datasets/Titanic-Dataset.csv')     # .. means: "Go one folder up (to the parent directory)"

In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [94]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### IMPORTANT NOTE
| Path          | Meaning                            |
| ------------- | ---------------------------------- |
| `.`           | Current folder (`Notes`)           |
| `..`          | One folder up (`Machine Learning`) |
| `../Datasets` | Go up, then into `Datasets`        |
| `../../`      | Go up **two levels**               |


### Step 2: Check for Duplicate Rows

- df.duplicated(): Returns a boolean Series indicating duplicate rows.

In [95]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
886    False
887    False
888    False
889    False
890    False
Length: 891, dtype: bool

### Step 3: Identify Column Data Types

- List comprehension with .dtype attribute to separate categorical and numerical columns.

- object dtype: Generally used for text or categorical data.

In [96]:
# This code loops through all the columns in the DataFrame and checks their data types. 
# Columns with data type 'object' are grouped as categorical columns, while all others are treated as numerical columns. 
# It then prints both lists to show which columns fall into each category.

cat_col = [col for col in df.columns if df[col].dtype == 'object']
num_col = [col for col in df.columns if df[col].dtype != 'object']

print('Categorical columns:', cat_col)
print('Numerical columns:', num_col)

Categorical columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
Numerical columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


#### Better Professional Approach

- Using pandas built-in method:

In [97]:
# This code uses pandas’ built-in select_dtypes() to automatically separate columns based on their data types. 
# It selects categorical columns by including 'object' types and numerical columns by including 'int64' and 'float64'. 
# Finally, it prints both groups to show how the dataset is divided.

cat_col = df.select_dtypes(include=['object']).columns
num_col = df.select_dtypes(include=['int64', 'float64']).columns

print('Categorical columns:', cat_col)
print('Numerical columns:', num_col)


Categorical columns: Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='object')
Numerical columns: Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')


#### Best Practice

In [98]:
# This code uses select_dtypes() to group columns more robustly by their data types. 
# It treats both 'object' and 'category' as categorical columns, while 'number' captures all numeric types (integers and floats) as numerical columns. 
# It then prints both groups for easy inspection.

cat_col = df.select_dtypes(include=['object', 'category']).columns
num_col = df.select_dtypes(include=['number']).columns

print('Categorical columns:', cat_col)
print('Numerical columns:', num_col)

Categorical columns: Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='object')
Numerical columns: Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')


### Step 4: Count Unique Values in the Categorical Columns

- df[cat_col].nunique(): Returns count of unique values per column.

In [99]:
# This code selects only the categorical columns (cat_col) from the DataFrame and applies .nunique() to them. 
# It returns the number of unique values in each categorical column, helping you understand how many distinct categories each feature has. 
# This is useful for identifying columns with high or low cardinality before encoding.

df[cat_col].nunique()

Name        891
Sex           2
Ticket      681
Cabin       147
Embarked      3
dtype: int64

### Step 5: Calculate Missing Values as Percentage

- df.isnull(): Detects missing values, returning boolean DataFrame.

- Sum missing across columns, normalize by total rows and multiply by 100.

In [100]:
# This code calculates the percentage of missing values in each column of the DataFrame. 
# It first identifies missing values using isnull(), sums them per column, divides by the total number of rows (df.shape[0]), and multiplies by 100 to get percentages. 
# The round(..., 2) ensures the results are displayed with 2 decimal places for readability.

round((df.isnull().sum() / df.shape[0]) * 100, 2)

PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

#### Breakdown of the above code

`df.isnull().sum()` : Creates a boolean mask where missing values are True (counted as 1) and valid values are False (0). The sum() method then adds these up to give the total number of missing values per column.

`/ df.shape[0]` : Divides the missing count by the total number of rows (the first element of DataFrame.shape) to get the fraction of missing data.

`* 100` : Converts the decimal fraction into a percentage.

`round(..., 2)` : Rounds the resulting percentages to two decimal places for better readability. 


### Step 6: Drop Irrelevant or Data-Heavy Missing Columns

- df.drop(columns=[]): Drops specified columns from the DataFrame.

- df.dropna(subset=[]): Removes rows where specified columns have missing values.

- fillna(): Fills missing values with specified value (e.g., mean).

In [101]:
# This code cleans the dataset by handling unnecessary columns and missing values. 
# It first drops irrelevant or high-missing columns (Name, Ticket, Cabin), then removes rows where the Embarked column is missing, 
# and finally fills missing values in the Age column using the mean age. 
# This ensures the dataset is more complete and ready for analysis or modeling.

df1 = df.drop(columns=['Name', 'Ticket', 'Cabin'])
df1.dropna(subset=['Embarked'], inplace=True)
df1['Age'] = df1['Age'].fillna(df1['Age'].mean())

This above code snippet performs standard data preprocessing steps, commonly used on datasets like the Titanic dataset, to clean data for machine learning models. 

Here is a breakdown of what each line does:

`1. df1 = df.drop(columns=['Name', 'Ticket', 'Cabin'])`

- Purpose: Removes specific columns from the DataFrame df.

- Details: It creates a new DataFrame df1 that excludes 'Name', 'Ticket', and 'Cabin'. These are often dropped because they are unique identifiers, have too many missing values ('Cabin'), or are not directly numerical.

Note: The original df remains unchanged. 

`2. df1.dropna(subset=['Embarked'], inplace=True)`

- Purpose: Removes rows where the 'Embarked' column has missing values ().

- Details: subset=['Embarked'] ensures rows are only dropped if the missing data is in the 'Embarked' column. inplace=True modifies df1 directly rather than creating a new copy. 

`3. df1['Age'] = df1['Age'].fillna(df1['Age'].mean())`

- Purpose: Fills missing values in the 'Age' column with the mean age of the dataset.

- Details: df1['Age'].mean() calculates the average of the existing ages, and fillna() replaces any values in that column with that average. 

#### Summary of Result

After these three lines, df1 is a cleaned DataFrame with:

- Unnecessary columns removed.

- Rows with missing 'Embarked' values removed.

- Missing 'Age' values imputed with the mean age. 